# Model 2: Qwen2.5-3B + RAG (LangChain + FAISS + SQuAD v2 KB)
## RAG-Augmented QA with ROUGE, BERTScore & Hallucination Evaluation

### Design
- **Fine-tuned on**: SQuAD v2 (baseline.ipynb)
- **RAG Knowledge Base**: SQuAD v2 training contexts (same domain as fine-tune data)
- **Evaluated on**: SQuAD v2 validation split — **same as Model 1**

### Why this is the correct comparison
Both models are evaluated on identical questions with identical gold answers.
The only variable is whether the model receives retrieved context (RAG) or not.
This gives a clean, controlled ablation: **does RAG help the fine-tuned model?**

```
SQuAD v2 Val Question
        ↓
   FAISS Retriever (SQuAD v2 Training Contexts KB)
        ↓ top-5 passages (after reranking)
   Fine-tuned Qwen2.5-3B
        ↓
   Answer → ROUGE + BERTScore + Faithfulness vs gold answer
```

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"
print("✅ CUDA debug flags set")

In [ ]:
import torch
import transformers, bitsandbytes, peft, trl

print("📦 Package versions:")
print(f"   torch          : {torch.__version__}")
print(f"   transformers   : {transformers.__version__}")
print(f"   bitsandbytes   : {bitsandbytes.__version__}")
print(f"   peft           : {peft.__version__}")
print(f"   trl            : {trl.__version__}")
print(f"   CUDA           : {torch.version.cuda}")
print()
if torch.cuda.is_available():
    print(f"   Device : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ All checks passed!")

In [ ]:
import os, gc, json, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder

# ── CONFIG ─────────────────────────────────────────────────────────────────────
BASE_MODEL          = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR         = "/content/qwen25-squadv2"
MERGED_DIR          = "/content/qwen25-squadv2-merged"
FAISS_PATH          = "./faiss_squadv2_bge"          # SQuAD v2 KB index
MODEL1_METRICS_PATH = "./model1_outputs/model1_metrics.json"
EMBED_MODEL         = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL      = "cross-encoder/ms-marco-MiniLM-L-6-v2"
SAVE_DIR            = "./model2_outputs"

EVAL_SAMPLES  = 500     # SQuAD v2 validation samples to evaluate on
KB_SAMPLES    = 80000   # SQuAD v2 training samples to index into KB
TOP_K         = 5
TOP_K_RERANK  = 10
MAX_SEQ_LEN   = 512
SEED          = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

print(f"🖥️  Device : {device} — {torch.cuda.get_device_name(0)}")
print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
print(f"   Fine-tune base : SQuAD v2 (from baseline.ipynb)")
print(f"   RAG KB         : SQuAD v2 training contexts ({KB_SAMPLES:,} samples)")
print(f"   Eval set       : SQuAD v2 validation ({EVAL_SAMPLES} samples)")
print(f"   Embed model    : {EMBED_MODEL}")
print(f"   Reranker       : {RERANKER_MODEL}")
print(f"   Top-K (fetch)  : {TOP_K_RERANK}  →  Top-K (reranked): {TOP_K}")
print("\n✅ Config ready!")

## Load SQuAD v2 — Build RAG Knowledge Base

In [ ]:
print("📥 Loading SQuAD v2...")
squad = load_dataset("rajpurkar/squad_v2")

for split, d in squad.items():
    print(f"   {split:12s}: {len(d):,} samples")

ex = squad["train"][0]
print(f"\n🔍 Sample question : {ex['question']}")
print(f"   Context (200 chars): {ex['context'][:200]}...")
print(f"   Answer : {ex['answers']['text'][0] if ex['answers']['text'] else '(unanswerable)'}")

In [ ]:
print(f"\n🔨 Building Knowledge Base from SQuAD v2 training contexts...")
print(f"   Using {KB_SAMPLES:,} training samples → extracting unique context passages")

docs    = []
skipped = 0
seen_contexts = set()    # SQuAD has duplicate contexts across questions — deduplicate

for sample in tqdm(squad["train"].select(range(KB_SAMPLES)), desc="Building KB"):
    context  = sample["context"].strip()
    question = sample["question"]
    answers  = sample["answers"]["text"]
    answer   = answers[0] if answers else "unanswerable"

    if not context or len(context) < 50:
        skipped += 1
        continue

    # Index each unique context once; include a representative Q&A for retrieval signal
    if context not in seen_contexts:
        seen_contexts.add(context)
        docs.append(Document(
            page_content=context[:1000],
            metadata={"source": "SQuAD_v2", "question": question[:80], "answer": answer[:80]}
        ))

    # Also store short question+context chunks for better retrieval
    chunk = f"Q: {question}\nContext: {context[:600]}"
    docs.append(Document(
        page_content=chunk,
        metadata={"source": "SQuAD_v2_qa", "question": question[:80]}
    ))

print(f"\n   ✅ {len(docs):,} documents built ({len(seen_contexts):,} unique contexts)")
print(f"   ⏭️  {skipped:,} passages skipped (too short)")
print(f"\n📄 Sample document (context chunk):")
print(docs[0].page_content[:400])

## Build FAISS Vector Store

In [ ]:
import shutil

print(f"📥 Loading embedding model: {EMBED_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={
        "normalize_embeddings": True,   # required for BGE models
        "batch_size": 128,
    },
)
print("✅ Embedding model ready (dim=768 for BGE-base)")

if os.path.exists(FAISS_PATH):
    print(f"\n📂 Loading existing FAISS index from {os.path.abspath(FAISS_PATH)}...")
    vectorstore = FAISS.load_local(
        FAISS_PATH, embeddings, allow_dangerous_deserialization=True,
    )
    print("✅ FAISS index loaded!")
else:
    print(f"\n🔨 Building FAISS index from {len(docs):,} documents...")
    t0 = datetime.now()
    vectorstore = FAISS.from_documents(docs, embeddings)
    print(f"   Built in {datetime.now()-t0}")
    vectorstore.save_local(FAISS_PATH)
    print(f"   💾 Saved → {os.path.abspath(FAISS_PATH)}")

# ── Reranker ──────────────────────────────────────────────────────────────────
print(f"\n📥 Loading reranker: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL, device=device, max_length=512)
print("✅ Reranker ready")

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RERANK})

# ── Retrieval test ─────────────────────────────────────────────────────────────
test_q     = "When did the Norman conquest of England occur?"
candidates = vectorstore.similarity_search(test_q, k=TOP_K_RERANK)
pairs      = [[test_q, doc.page_content] for doc in candidates]
scores     = reranker.predict(pairs)
reranked   = [doc for _, doc in sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)][:TOP_K]

print(f"\n🔍 Retrieval test: '{test_q}'")
print(f"   Before rerank — top doc: {candidates[0].page_content[:150]}...")
print(f"   After  rerank — top doc: {reranked[0].page_content[:150]}...")
print(f"\n✅ Retriever + Reranker ready — fetching top {TOP_K_RERANK}, reranking to {TOP_K}")

## Load Fine-Tuned Model (from baseline.ipynb)

In [ ]:
from transformers import BitsAndBytesConfig
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free before loading: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

if os.path.exists(MERGED_DIR):
    print(f"📥 Loading pre-merged fine-tuned model from {os.path.abspath(MERGED_DIR)}...")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        quantization_config=bnb_config,
        device_map="cuda:0",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model_source = "MERGED_DIR (fine-tuned on SQuAD v2)"
    print("   ✅ Pre-merged model loaded")

elif os.path.exists(ADAPTER_DIR):
    print(f"📥 Loading base model + LoRA adapters from {os.path.abspath(ADAPTER_DIR)}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model = model.merge_and_unload()
    model = model.to(torch.float16)
    model_source = "ADAPTER_DIR (fine-tuned on SQuAD v2, merged on-the-fly)"
    print("   ✅ Adapters merged into base model")

else:
    raise FileNotFoundError(
        f"❌ No fine-tuned model found!\n"
        f"   Expected MERGED_DIR  : {os.path.abspath(MERGED_DIR)}\n"
        f"   Expected ADAPTER_DIR : {os.path.abspath(ADAPTER_DIR)}\n"
        f"   Please run baseline.ipynb first!"
    )

model.config.use_cache = True
model.eval()

print(f"\n✅ Model ready!")
print(f"   Source    : {model_source}")
print(f"   VRAM used : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM free : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"   dtype     : {next(model.parameters()).dtype}")

## RAG Prompt & Inference Pipeline

In [ ]:
from transformers import pipeline as hf_pipeline

gen_pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)
print("✅ Generation pipeline ready!")

RAG_SYSTEM = (
    "You are a helpful and precise question-answering assistant. "
    "You are given retrieved passages that may contain relevant information. "
    "Use the context to answer the question accurately and concisely. "
    "If the context does not support an answer, say 'unanswerable'."
)

def retrieve_context(question, k_fetch=TOP_K_RERANK, k_final=TOP_K):
    candidates = vectorstore.similarity_search(question, k=k_fetch)
    pairs      = [[question, doc.page_content] for doc in candidates]
    scores     = reranker.predict(pairs)
    ranked     = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs   = [doc for _, doc in ranked[:k_final]]
    parts = []
    for i, doc in enumerate(top_docs, 1):
        parts.append(f"[Passage {i}]\n{doc.page_content}")
    return "\n\n".join(parts)

def generate_rag_answer(question, max_new_tokens=150):
    context = retrieve_context(question)
    prompt = (
        f"<|im_start|>system\n{RAG_SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"--- RETRIEVED PASSAGES ---\n{context}\n--- END PASSAGES ---\n\n"
        f"Question: {question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    out      = gen_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                        pad_token_id=tokenizer.eos_token_id)
    full     = out[0]["generated_text"]
    response = full.split("<|im_start|>assistant")[-1]
    response = response.replace("<|im_end|>", "").strip()
    return response, context

# Smoke test on a SQuAD v2 validation sample
print("\n🧪 Testing RAG pipeline...")
test_sample = squad["validation"][0]
ans, ctx    = generate_rag_answer(test_sample["question"])
gold        = test_sample["answers"]["text"][0] if test_sample["answers"]["text"] else "unanswerable"
print(f"Q       : {test_sample['question']}")
print(f"Context : {ctx[:200]}...")
print(f"Answer  : {ans[:200]}")
print(f"Gold    : {gold}")
print("\n✅ RAG pipeline works!")

## Full Inference on SQuAD v2 Validation Set

In [ ]:
print(f"🔮 RAG inference — {EVAL_SAMPLES} SQuAD v2 validation samples (top-{TOP_K} passages each)...")

rag_results = []
val_subset  = squad["validation"].select(range(EVAL_SAMPLES))

t0 = datetime.now()
for i, sample in enumerate(tqdm(val_subset, desc="RAG Inference")):
    question  = sample["question"]
    answers   = sample["answers"]["text"]
    reference = answers[0] if answers else "unanswerable"
    answerable = len(answers) > 0
    generated, retrieved = generate_rag_answer(question)

    rag_results.append({
        "id":                i,
        "question":          question,
        "generated":         generated,
        "reference":         reference,
        "retrieved_context": retrieved,
        "answerable":        answerable,
    })

rag_df = pd.DataFrame(rag_results)
print(f"\n✅ Done in {datetime.now()-t0}!")
print(f"   Total samples      : {len(rag_df)}")
print(f"   Answerable         : {rag_df['answerable'].sum()}")
print(f"   Unanswerable       : {(~rag_df['answerable']).sum()}")
print(f"\nSample:")
print(f"Q        : {rag_df.iloc[0]['question']}")
print(f"Gold     : {rag_df.iloc[0]['reference']}")
print(f"Generated: {rag_df.iloc[0]['generated'][:200]}")

## ROUGE Evaluation

In [ ]:
scorer     = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in tqdm(rag_df.iterrows(), total=len(rag_df), desc="ROUGE"):
    s = scorer.score(row["reference"], row["generated"].strip() or " ")
    r1.append(s["rouge1"].fmeasure)
    r2.append(s["rouge2"].fmeasure)
    rL.append(s["rougeL"].fmeasure)

rag_df["rouge1"] = r1
rag_df["rouge2"] = r2
rag_df["rougeL"] = rL

# Exact match
rag_df["exact_match"] = rag_df.apply(
    lambda row: int(row["generated"].strip().lower() == row["reference"].strip().lower()), axis=1
)

print(f"\n{'='*52}")
print(f"  📊 ROUGE — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*52}")
print(f"  ROUGE-1 : {np.mean(r1):.4f}  ±{np.std(r1):.4f}")
print(f"  ROUGE-2 : {np.mean(r2):.4f}  ±{np.std(r2):.4f}")
print(f"  ROUGE-L : {np.mean(rL):.4f}  ±{np.std(rL):.4f}")
print(f"  Exact Match : {rag_df['exact_match'].mean()*100:.2f}%")
print(f"{'='*52}")

In [ ]:
pip install bert-score

In [ ]:
from bert_score import score as bert_score_fn

print("🔮 Computing BERTScore (semantic similarity vs gold answers)...")
print("   Using model: microsoft/deberta-xlarge-mnli")

bs_candidates = rag_df["generated"].apply(lambda x: x.strip() or " ").tolist()
bs_references = rag_df["reference"].tolist()

P, R, F1 = bert_score_fn(
    bs_candidates,
    bs_references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    batch_size=16,
    device=device,
    verbose=True,
)

rag_df["bertscore_P"]  = P.numpy()
rag_df["bertscore_R"]  = R.numpy()
rag_df["bertscore_F1"] = F1.numpy()

mean_P  = P.mean().item()
mean_R  = R.mean().item()
mean_F1 = F1.mean().item()

print(f"\n{'='*56}")
print(f"  📊 BERTScore — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Precision : {mean_P:.4f}  ±{P.std().item():.4f}")
print(f"  Recall    : {mean_R:.4f}  ±{R.std().item():.4f}")
print(f"  F1        : {mean_F1:.4f}  ±{F1.std().item():.4f}")
print(f"{'='*56}")

## Faithfulness / Hallucination Evaluation

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("🧠 Loading NLI model...")
nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device, max_length=512)
print("✅ NLI model ready")

def faith_batch(premises, hypotheses, bs=32):
    scores = []
    for i in range(0, len(premises), bs):
        pairs  = list(zip(premises[i:i+bs], hypotheses[i:i+bs]))
        logits = nli.predict(pairs, apply_softmax=True)
        scores.extend(logits[:, 1].tolist())
    return scores

# Faithfulness vs gold reference answer
print("\n🔮 Computing faithfulness (generated vs gold answer)...")
ref_scores = faith_batch(rag_df["reference"].tolist(), rag_df["generated"].tolist())
rag_df["faith_score"] = ref_scores
rag_df["hallucinated"] = rag_df["faith_score"] < 0.3

# Source grounding vs retrieved context
print("🔮 Computing grounding (generated vs retrieved passages)...")
trunc_ctx  = rag_df["retrieved_context"].apply(lambda x: x[:500]).tolist()
src_scores = faith_batch(trunc_ctx, rag_df["generated"].tolist())
rag_df["src_faith"]  = src_scores
rag_df["src_halluc"] = rag_df["src_faith"] < 0.3

mean_f    = np.mean(ref_scores)
halluc    = rag_df["hallucinated"].mean()
mean_src  = np.mean(src_scores)
halluc_src = rag_df["src_halluc"].mean()

# Per-answerability breakdown
ans_df   = rag_df[rag_df["answerable"] == True]
unans_df = rag_df[rag_df["answerable"] == False]

print(f"\n{'='*56}")
print(f"  🧠 FAITHFULNESS — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Reference faithfulness (vs gold answer):")
print(f"    Mean score        : {mean_f:.4f}")
print(f"    Faithful  (≥0.3)  : {(1-halluc)*100:.1f}%")
print(f"    Hallucinated(<0.3): {halluc*100:.1f}%")
print(f"  Source faithfulness (vs retrieved context):")
print(f"    Mean score        : {mean_src:.4f}")
print(f"    Grounded  (≥0.3)  : {(1-halluc_src)*100:.1f}%")
print(f"    Not grounded(<0.3): {halluc_src*100:.1f}%")
print(f"  Answerable   hallucination rate: {ans_df['hallucinated'].mean()*100:.1f}%")
print(f"  Unanswerable hallucination rate: {unans_df['hallucinated'].mean()*100:.1f}%")
print(f"{'='*56}")

## Visualisation — Model 2 (RAG)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle("Model 2: Qwen2.5-3B + RAG (SQuAD v2 KB) — Full Evaluation",
             fontsize=15, fontweight="bold")

# [0,0] ROUGE bars
ax = axes[0, 0]
means = [np.mean(r1), np.mean(r2), np.mean(rL)]
stds  = [np.std(r1),  np.std(r2),  np.std(rL)]
bars  = ax.bar(["ROUGE-1","ROUGE-2","ROUGE-L"], means, yerr=stds,
               color=["#2196F3","#4CAF50","#FF9800"], capsize=5, edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("ROUGE Scores", fontweight="bold"); ax.set_ylabel("F1 Score")
ax.axhline(0.4, color="red", linestyle="--", alpha=0.5, label="Good threshold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars, means):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)

# [0,1] Faithfulness histogram
ax = axes[0, 1]
ax.hist(ref_scores, bins=20, color="#9C27B0", edgecolor="black", alpha=0.8)
ax.axvline(0.3,    color="red",  linestyle="--", lw=2, label="Hallucination threshold")
ax.axvline(mean_f, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_f:.3f}")
ax.set_title("Faithfulness Distribution", fontweight="bold")
ax.set_xlabel("Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [0,2] Faithful vs Hallucinated pie
ax = axes[0, 2]
ax.pie([1-halluc, halluc],
       labels=[f"Faithful\n{(1-halluc)*100:.1f}%", f"Hallucinated\n{halluc*100:.1f}%"],
       colors=["#4CAF50","#F44336"], autopct="%1.1f%%", startangle=90,
       wedgeprops={"edgecolor":"white","linewidth":2})
ax.set_title("Faithful vs Hallucinated", fontweight="bold")

# [1,0] Hallucination by answerability
ax = axes[1, 0]
cats = ["Answerable", "Unanswerable"]
halluc_rates = [ans_df["hallucinated"].mean()*100, unans_df["hallucinated"].mean()*100]
bars2 = ax.bar(cats, halluc_rates, color=["#2196F3","#F44336"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title("Hallucination by Answerability", fontweight="bold")
ax.set_ylabel("Hallucination Rate (%)"); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars2, halluc_rates):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)

# [1,1] All metrics summary
ax = axes[1, 1]
mnames  = ["Exact Match", "ROUGE-L", "ROUGE-1", "Faithful", "BERTScore F1"]
mvalues = [rag_df["exact_match"].mean()*100, np.mean(rL)*100, np.mean(r1)*100,
           (1-halluc)*100, mean_F1*100]
bars3 = ax.bar(mnames, mvalues,
               color=["#FF5722","#FF9800","#2196F3","#4CAF50","#673AB7"],
               edgecolor="black", alpha=0.85)
ax.set_ylim(0, 100); ax.set_title("All Metrics (%)", fontweight="bold")
ax.set_ylabel("Score (%)"); ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", labelsize=8)
for b, v in zip(bars3, mvalues):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)

# [1,2] ROUGE-1 by answerability
ax = axes[1, 2]
ans_r1   = rag_df[rag_df["answerable"]==True]["rouge1"].mean()
unans_r1 = rag_df[rag_df["answerable"]==False]["rouge1"].mean()
ax.bar(["Answerable\nROUGE-1", "Unanswerable\nROUGE-1"], [ans_r1, unans_r1],
       color=["#00BCD4","#FF9800"], edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("ROUGE-1 by Answerability", fontweight="bold")
ax.set_ylabel("ROUGE-1 F1"); ax.grid(axis="y", alpha=0.3)
for i, v in enumerate([ans_r1, unans_r1]):
    ax.text(i, v+0.01, f"{v:.3f}", ha="center", fontweight="bold", fontsize=12)

# [2,0] BERTScore P/R/F1 bars
ax = axes[2, 0]
bs_means = [mean_P, mean_R, mean_F1]
bs_stds  = [P.std().item(), R.std().item(), F1.std().item()]
bs_bars  = ax.bar(["Precision","Recall","F1"], bs_means, yerr=bs_stds,
                  color=["#3F51B5","#00BCD4","#673AB7"],
                  capsize=5, edgecolor="black", alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("BERTScore (deberta-xlarge-mnli)", fontweight="bold")
ax.set_ylabel("Score")
ax.axhline(0.85, color="red", linestyle="--", alpha=0.5, label="Good threshold (≥0.85)")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bs_bars, bs_means):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.4f}", ha="center", fontweight="bold", fontsize=10)

# [2,1] BERTScore F1 distribution
ax = axes[2, 1]
ax.hist(F1.numpy(), bins=25, color="#673AB7", edgecolor="black", alpha=0.8)
ax.axvline(mean_F1, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_F1:.4f}")
ax.axvline(0.85,    color="red",  linestyle="--", lw=2, label="Good threshold (0.85)")
ax.set_title("BERTScore F1 Distribution", fontweight="bold")
ax.set_xlabel("BERTScore F1"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [2,2] Grounding vs retrieved context
ax = axes[2, 2]
ax.hist(src_scores, bins=20, color="#00BCD4", edgecolor="black", alpha=0.8)
ax.axvline(0.3,      color="red",  linestyle="--", lw=2, label="Grounding threshold")
ax.axvline(mean_src, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_src:.3f}")
ax.set_title("Grounding vs Retrieved Context", fontweight="bold")
ax.set_xlabel("Source Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_results.png")

## Model 1 vs Model 2 — Final Comparison

In [ ]:
try:
    with open(MODEL1_METRICS_PATH) as f:
        m1 = json.load(f)
    m1_r1, m1_r2, m1_rL = m1["rouge"]["rouge1"], m1["rouge"]["rouge2"], m1["rouge"]["rougeL"]
    m1_faith  = m1["faithfulness"]["mean_score"]
    m1_halluc = m1["faithfulness"]["hallucination_rate"]
    m1_bs_f1  = m1.get("bertscore", {}).get("f1", None)
    m1_em     = m1.get("exact_match", None)
    print(f"✅ Loaded Model 1 metrics from {os.path.abspath(MODEL1_METRICS_PATH)}")
except FileNotFoundError:
    print(f"⚠️  Not found: {os.path.abspath(MODEL1_METRICS_PATH)}")
    print("   Run baseline.ipynb first — using placeholders for display")
    m1_r1, m1_r2, m1_rL = 0.35, 0.18, 0.30
    m1_faith, m1_halluc  = 0.45, 0.45
    m1_bs_f1, m1_em      = None, None

m2_r1, m2_r2, m2_rL = np.mean(r1), np.mean(r2), np.mean(rL)
m2_faith  = mean_f
m2_halluc = halluc
m2_bs_f1  = mean_F1
m2_em     = rag_df["exact_match"].mean()

print(f"\n{'='*66}")
print(f"  📊 FINAL COMPARISON: Model 1 (No RAG) vs Model 2 (RAG)")
print(f"  Both evaluated on SQuAD v2 validation — clean controlled comparison")
print(f"  Fine-tune: SQuAD v2  |  RAG KB: SQuAD v2 training contexts")
print(f"{'='*66}")
print(f"  {'Metric':<30} {'Model 1':>9} {'Model 2':>9} {'Δ (M2-M1)':>10}")
print(f"  {'─'*60}")

for label, v1, v2 in [
    ("ROUGE-1",            m1_r1,    m2_r1),
    ("ROUGE-2",            m1_r2,    m2_r2),
    ("ROUGE-L",            m1_rL,    m2_rL),
    ("Faithfulness Score", m1_faith, m2_faith),
]:
    print(f"  {label:<30} {v1:>9.4f} {v2:>9.4f} {v2-v1:>+10.4f}")

if m1_bs_f1 is not None:
    print(f"  {'BERTScore F1':<30} {m1_bs_f1:>9.4f} {m2_bs_f1:>9.4f} {m2_bs_f1-m1_bs_f1:>+10.4f}")
else:
    print(f"  {'BERTScore F1':<30} {'N/A':>9} {m2_bs_f1:>9.4f} {'(no M1 baseline)':>10}")

if m1_em is not None:
    print(f"  {'Exact Match':<30} {m1_em*100:>8.1f}% {m2_em*100:>8.1f}% {(m2_em-m1_em)*100:>+9.1f}%")

print(f"  {'─'*60}")
print(f"  {'Hallucination Rate':<30} {m1_halluc*100:>8.1f}% {m2_halluc*100:>8.1f}% {(m2_halluc-m1_halluc)*100:>+9.1f}%")
print(f"{'='*66}")
print(f"  RAG source grounding score  : {mean_src:.4f}")
print(f"  ↑ ROUGE/Faithfulness/BERTScore = better | ↓ Hallucination = better")

## Visualisation — Full Comparison

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Model 1 (No RAG) vs Model 2 (RAG) — SQuAD v2 Validation — Controlled Comparison",
             fontsize=14, fontweight="bold")

# [0,0] ROUGE grouped bars
ax = axes[0, 0]
x   = np.arange(3)
w   = 0.35
rouge_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
m1_vals = [m1_r1, m1_r2, m1_rL]
m2_vals = [m2_r1, m2_r2, m2_rL]
b1 = ax.bar(x - w/2, m1_vals, w, label="Model 1 (No RAG)", color="#B0BEC5", edgecolor="black", alpha=0.85)
b2 = ax.bar(x + w/2, m2_vals, w, label="Model 2 (RAG)",    color="#2196F3", edgecolor="black", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(rouge_labels)
ax.set_ylim(0, 1); ax.set_title("ROUGE Scores: M1 vs M2", fontweight="bold")
ax.set_ylabel("F1 Score"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(list(b1)+list(b2), m1_vals+m2_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")

# [0,1] BERTScore F1 grouped bars
ax = axes[0, 1]
bs_m1 = m1_bs_f1 if m1_bs_f1 is not None else 0
bs_m2 = m2_bs_f1
b1 = ax.bar([0 - 0.2], [bs_m1], 0.35, label="Model 1 (No RAG)", color="#B0BEC5", edgecolor="black", alpha=0.85)
b2 = ax.bar([0 + 0.2], [bs_m2], 0.35, label="Model 2 (RAG)",    color="#673AB7", edgecolor="black", alpha=0.85)
ax.set_xlim(-0.6, 0.6); ax.set_xticks([0]); ax.set_xticklabels(["BERTScore F1"])
ax.set_ylim(0, 1); ax.set_title("BERTScore F1: M1 vs M2", fontweight="bold")
ax.set_ylabel("Score"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in [(b1[0], bs_m1), (b2[0], bs_m2)]:
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.4f}", ha="center", fontweight="bold", fontsize=11)

# [0,2] Hallucination rate
ax = axes[0, 2]
cats  = ["Model 1\n(No RAG)", "Model 2\n(RAG)"]
hvals = [m1_halluc*100, m2_halluc*100]
bars2 = ax.bar(cats, hvals, color=["#F44336","#FF9800"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title("Hallucination Rate (lower = better)", fontweight="bold")
ax.set_ylabel("Hallucination Rate (%)"); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars2, hvals):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)

# [1,0] Faithfulness comparison
ax = axes[1, 0]
fvals = [m1_faith, m2_faith]
bars3 = ax.bar(cats, fvals, color=["#B0BEC5","#4CAF50"], edgecolor="black", alpha=0.85, width=0.4)
ax.set_ylim(0, 1); ax.set_title("Faithfulness Score: M1 vs M2", fontweight="bold")
ax.set_ylabel("Mean Faithfulness")
ax.axhline(0.3, color="red", linestyle="--", alpha=0.5, label="Threshold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
for b, v in zip(bars3, fvals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.4f}", ha="center", fontweight="bold", fontsize=12)

# [1,1] Grounding vs retrieved context (RAG-only metric)
ax = axes[1, 1]
ax.hist(src_scores, bins=20, color="#00BCD4", edgecolor="black", alpha=0.8)
ax.axvline(0.3,      color="red",  linestyle="--", lw=2, label="Grounding threshold")
ax.axvline(mean_src, color="blue", linestyle="-",  lw=2, label=f"Mean = {mean_src:.3f}")
ax.set_title("Model 2: Grounding vs Retrieved Context", fontweight="bold")
ax.set_xlabel("Source Faithfulness Score"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# [1,2] Delta summary (M2 - M1)
ax = axes[1, 2]
delta_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "Faithfulness", "BERTScore F1"]
bs_delta = (m2_bs_f1 - m1_bs_f1) if m1_bs_f1 is not None else 0
deltas   = [m2_r1-m1_r1, m2_r2-m1_r2, m2_rL-m1_rL, m2_faith-m1_faith, bs_delta]
colors   = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars4    = ax.bar(delta_labels, deltas, color=colors, edgecolor="black", alpha=0.85)
ax.axhline(0, color="black", lw=1)
ax.set_title("Δ RAG Improvement (M2 − M1)", fontweight="bold")
ax.set_ylabel("Delta Score"); ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", labelsize=8)
for b, v in zip(bars4, deltas):
    ypos = v + 0.002 if v >= 0 else v - 0.008
    ax.text(b.get_x()+b.get_width()/2, ypos, f"{v:+.4f}", ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_comparison.png")

## Save Results & Metrics

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
rag_df.to_csv(f"{SAVE_DIR}/model2_results.csv", index=False, escapechar="\\")

model2_metrics = {
    "model":            "Qwen2.5-3B-Instruct (Fine-tuned on SQuAD v2 + RAG SQuAD v2 KB)",
    "eval_samples":     EVAL_SAMPLES,
    "kb_samples":       KB_SAMPLES,
    "eval_dataset":     "SQuAD v2 validation",
    "kb_dataset":       "SQuAD v2 training contexts",
    "finetune_dataset": "SQuAD v2",
    "rouge": {
        "rouge1":     round(float(np.mean(r1)), 4),
        "rouge2":     round(float(np.mean(r2)), 4),
        "rougeL":     round(float(np.mean(rL)), 4),
        "rouge1_std": round(float(np.std(r1)),  4),
        "rouge2_std": round(float(np.std(r2)),  4),
        "rougeL_std": round(float(np.std(rL)),  4),
    },
    "exact_match": round(float(rag_df["exact_match"].mean()), 4),
    "bertscore": {
        "precision":     round(mean_P,         4),
        "recall":        round(mean_R,         4),
        "f1":            round(mean_F1,        4),
        "precision_std": round(P.std().item(), 4),
        "recall_std":    round(R.std().item(), 4),
        "f1_std":        round(F1.std().item(),4),
        "model":         "microsoft/deberta-xlarge-mnli",
    },
    "faithfulness": {
        "mean_score":          round(float(mean_f),    4),
        "hallucination_rate":  round(float(halluc),    4),
        "faithful_pct":        round((1-halluc)*100,   2),
        "halluc_pct":          round(halluc*100,       2),
        "source_mean":         round(float(mean_src),  4),
        "source_grounded_pct": round((1-halluc_src)*100, 2),
        "samples_evaluated":   int(len(rag_df)),
    },
}

with open(f"{SAVE_DIR}/model2_metrics.json", "w") as f:
    json.dump(model2_metrics, f, indent=2)

bs_delta = round(m2_bs_f1 - m1_bs_f1, 4) if m1_bs_f1 is not None else None
comparison = {
    "experiment_note": "Clean controlled comparison — both models evaluated on SQuAD v2 validation",
    "model1_no_rag": {
        "dataset":      "SQuAD v2 validation",
        "rouge1": m1_r1, "rouge2": m1_r2, "rougeL": m1_rL,
        "faithfulness": m1_faith, "hallucination": m1_halluc,
        "bertscore_f1": m1_bs_f1,
    },
    "model2_rag": {
        "finetune_dataset": "SQuAD v2",
        "kb_dataset":       "SQuAD v2 training contexts",
        "eval_dataset":     "SQuAD v2 validation",
        "rouge1":        round(m2_r1,    4),
        "rouge2":        round(m2_r2,    4),
        "rougeL":        round(m2_rL,    4),
        "faithfulness":  round(m2_faith, 4),
        "hallucination": round(m2_halluc,4),
        "bertscore_f1":  round(m2_bs_f1, 4),
    },
    "delta_model2_minus_model1": {
        "rouge1":        round(m2_r1    - m1_r1,    4),
        "rouge2":        round(m2_r2    - m1_r2,    4),
        "rougeL":        round(m2_rL    - m1_rL,    4),
        "faithfulness":  round(m2_faith - m1_faith, 4),
        "hallucination": round(m2_halluc- m1_halluc,4),
        "bertscore_f1":  bs_delta,
    },
    "note": "Positive delta = Model 2 better for ROUGE/faithfulness/BERTScore. Negative delta = Model 2 better for hallucination."
}

with open(f"{SAVE_DIR}/final_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print(f"💾 Saved to: {os.path.abspath(SAVE_DIR)}")
print(f"   model2_results.csv")
print(f"   model2_metrics.json")
print(f"   final_comparison.json")
print()
print("=" * 52)
print("  ✅ MODEL 2 COMPLETE — SUMMARY")
print("=" * 52)
print(f"  ROUGE-1       : {model2_metrics['rouge']['rouge1']:.4f}")
print(f"  ROUGE-2       : {model2_metrics['rouge']['rouge2']:.4f}")
print(f"  ROUGE-L       : {model2_metrics['rouge']['rougeL']:.4f}")
print(f"  BERTScore F1  : {model2_metrics['bertscore']['f1']:.4f}")
print(f"  Faithful      : {model2_metrics['faithfulness']['faithful_pct']:.1f}%")
print(f"  Hallucinate   : {model2_metrics['faithfulness']['halluc_pct']:.1f}%")
print(f"  Grounded      : {model2_metrics['faithfulness']['source_grounded_pct']:.1f}%")
print("=" * 52)